In [1]:
import numpy as np
import pandas as pd
import time

from sklearn.datasets import make_classification
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

from scipy.stats import randint

In [2]:
# Generate synthetic student placement dataset
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=15,
    n_redundant=5,
    n_classes=2,
    weights=[0.9, 0.1],   # 90% placed, 10% unplaced
    random_state=42
)

# Convert to DataFrame for better readability
feature_names = [
    'CGPA', 'Internships', 'Backlogs', 'Projects', 'Coding_Skills',
    'Communication', 'Attendance', 'Aptitude', 'Technical_Test',
    'Mock_Interview', 'DSA', 'Hackathons', 'Certifications',
    'Leadership', 'Teamwork', 'Problem_Solving', 'Discipline',
    'Research', 'Networking', 'Confidence'
]

df = pd.DataFrame(X, columns=feature_names)
df['Placement_Status'] = y

print(df.head())

        CGPA  Internships  Backlogs  Projects  Coding_Skills  Communication  \
0  -4.906442     3.442789  0.558964 -0.976764      -1.568805      -4.271982   
1  -8.460842    -0.463074 -3.253334 -1.909931       1.197232       0.553973   
2  -6.678971    -0.854743 -2.214812 -0.529275       2.562596      -0.864114   
3  10.465024     1.070944 -3.562432 -0.849062       2.183860      -0.609893   
4   5.599516    -1.776412 -1.304322 -0.720074       5.859373      -3.292432   

   Attendance  Aptitude  Technical_Test  Mock_Interview  ...  Hackathons  \
0   -3.727921  0.111868        2.119795       -2.522812  ...   -7.492478   
1   -2.769455  0.090651        1.968285        3.350884  ...    1.735225   
2   -1.020312  3.591929       -2.145187        3.366273  ...    3.676774   
3    0.946327 -1.046141       -2.057053       -2.056650  ...   -1.449095   
4    3.152205  7.099882       -3.321076        3.245486  ...    6.608729   

   Certifications  Leadership  Teamwork  Problem_Solving  Discipline

In [3]:
# Split data into training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

Training Shape: (800, 20)
Testing Shape: (200, 20)


In [4]:
scaler = StandardScaler()

# Fit only on training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data
X_test_scaled = scaler.transform(X_test)

Phase 2: The Baseline & The “Metric Trap”

In [5]:
baseline_model = RandomForestClassifier(random_state=42)

baseline_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_baseline = baseline_model.predict(X_test_scaled)

# Metrics
baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
baseline_f1 = f1_score(y_test, y_pred_baseline)

print("Baseline Accuracy:", baseline_accuracy)
print("Baseline F1-Score:", baseline_f1)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_baseline))

Baseline Accuracy: 0.915
Baseline F1-Score: 0.2608695652173913

Classification Report:

              precision    recall  f1-score   support

           0       0.91      1.00      0.95       180
           1       1.00      0.15      0.26        20

    accuracy                           0.92       200
   macro avg       0.96      0.57      0.61       200
weighted avg       0.92      0.92      0.89       200



In [6]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

rf_model = RandomForestClassifier(random_state=42)

# Grid Search for Accuracy
grid_accuracy = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)

start_time = time.time()

grid_accuracy.fit(X_train_scaled, y_train)

accuracy_time = time.time() - start_time

print("Best Parameters (Accuracy):")
print(grid_accuracy.best_params_)

print("Best Accuracy Score:")
print(grid_accuracy.best_score_)

Best Parameters (Accuracy):
{'max_depth': None, 'min_samples_split': 5, 'n_estimators': 50}
Best Accuracy Score:
0.915


In [7]:
# Grid Search for F1
grid_f1 = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1
)

start_time = time.time()

grid_f1.fit(X_train_scaled, y_train)

f1_time = time.time() - start_time

print("Best Parameters (F1):")
print(grid_f1.best_params_)

print("Best F1 Score:")
print(grid_f1.best_score_)

Best Parameters (F1):
{'max_depth': None, 'min_samples_split': 5, 'n_estimators': 50}
Best F1 Score:
0.300098731677679


In [8]:
best_acc_model = grid_accuracy.best_estimator_

y_pred_acc = best_acc_model.predict(X_test_scaled)

print("Test Accuracy:", accuracy_score(y_test, y_pred_acc))
print("Test F1:", f1_score(y_test, y_pred_acc))

Test Accuracy: 0.915
Test F1: 0.2608695652173913


In [9]:
best_f1_model = grid_f1.best_estimator_

y_pred_f1 = best_f1_model.predict(X_test_scaled)

print("Test Accuracy:", accuracy_score(y_test, y_pred_f1))
print("Test F1:", f1_score(y_test, y_pred_f1))

Test Accuracy: 0.915
Test F1: 0.2608695652173913
